# 04 - Advanced Topics: Video Tracking, Text-Prompted Segmentation & CLIP Fine-Tuning

## Going Beyond Single-Frame, Single-Model Pipelines

In Notebooks 01–03, we progressed from manual image processing through object detection to foundation models. This notebook pushes into **three cutting-edge workflows** that bridge the gap between workshop demos and production systems.

| Task | Models | What It Does |
|---|---|---|
| **Video Object Tracking** | SAM 2 Video Predictor | Click once on frame 1 → get pixel-perfect masks for the entire video |
| **Text-Prompted Segmentation** | Grounding DINO + SAM 2 | Type `"coffee mug"` → detect and segment it. No training, no class list |
| **Domain-Adapted Classification** | Fine-tuned CLIP | Teach CLIP your specialty — improve zero-shot accuracy on niche data |

> **Prerequisites:** Complete Notebooks 01–03 first. We reuse the SAM 2 checkpoint, helper functions, and sample files from those notebooks.

## Shared Setup

Load the SAM 2 checkpoint and helper functions we'll reuse across all three sections.

In [ ]:
import os, torch, cv2, numpy as np, matplotlib.pyplot as plt
from PIL import Image

# ── SAM 2 checkpoint (same as Notebook 03) ──
SAM2_CHECKPOINT = "models/sam2.1_hiera_small.pt"
SAM2_CONFIG = "configs/sam2.1/sam2.1_hiera_s.yaml"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ── Visualisation helpers ──
def show_mask(mask, ax, random_color=True, color=None):
    """Overlay a single binary mask on a Matplotlib axis."""
    if color is not None:
        rgba = np.array([*color, 0.5])
    elif random_color:
        rgba = np.concatenate([np.random.random(3), [0.5]])
    else:
        rgba = np.array([30/255, 144/255, 255/255, 0.5])
    h, w = mask.shape[-2:]
    ax.imshow(mask.reshape(h, w, 1) * rgba.reshape(1, 1, -1))

def show_box(box, ax, label=None, color='lime'):
    """Draw a bounding box on a Matplotlib axis."""
    x0, y0, x1, y1 = box
    rect = plt.Rectangle((x0, y0), x1 - x0, y1 - y0,
                          edgecolor=color, facecolor='none', linewidth=2)
    ax.add_patch(rect)
    if label:
        ax.text(x0, y0 - 6, label, color=color, fontsize=9,
                fontweight='bold', backgroundcolor='black')

def show_points(coords, labels, ax, marker_size=200):
    """Draw prompt points (green = foreground, red = background)."""
    pos = coords[labels == 1]
    neg = coords[labels == 0]
    ax.scatter(pos[:, 0], pos[:, 1], color='lime', marker='*',
               s=marker_size, edgecolors='white', linewidths=1.2, zorder=5)
    ax.scatter(neg[:, 0], neg[:, 1], color='red', marker='*',
               s=marker_size, edgecolors='white', linewidths=1.2, zorder=5)

print("Setup complete!")

---

## Task 1: SAM 2 Video Predictor — Track Objects Across a Video

In Notebook 03's Mega-Pipeline, we ran YOLO + SAM 2 **independently on every frame**. That works, but it's slow and doesn't maintain identity — the same object might get different masks from frame to frame.

**SAM 2's Video Predictor** solves this with a fundamentally different approach:

1. **You prompt once** — click a point (or draw a box) on a single frame.
2. **SAM 2 propagates** — it tracks that object through every subsequent frame using temporal attention, producing consistent masks without re-prompting.

This is the same technology behind Meta's video segmentation demos. Under the hood, SAM 2 maintains a **memory bank** of object appearances and uses cross-attention to match objects across frames.

### Workflow
```
Frame 1: You click a point on the car
         ↓
SAM 2 encodes the object appearance into memory
         ↓
Frames 2–N: SAM 2 attends to memory + current frame → mask
         ↓
Result: Consistent, pixel-perfect tracking with zero per-frame effort
```

### Step 1: Extract Video Frames

`SAM2VideoPredictor` expects frames as sequentially numbered JPEG files in a directory. We'll extract them from one of our sample videos.

In [ ]:
import shutil

# ── CONFIG ──
VIDEO_PATH = "samples/videos/car.mp4"
MAX_FRAMES = 120  # Limit frames for workshop speed (0 = all)
# ─────────────

FRAME_DIR = "results/video_frames"

# Clean and recreate frame directory
if os.path.exists(FRAME_DIR):
    shutil.rmtree(FRAME_DIR)
os.makedirs(FRAME_DIR, exist_ok=True)

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    # SAM 2 expects JPEG files named as sequential numbers
    cv2.imwrite(os.path.join(FRAME_DIR, f"{frame_count:05d}.jpg"), frame)
    frame_count += 1
    if MAX_FRAMES > 0 and frame_count >= MAX_FRAMES:
        break

cap.release()
print(f"Extracted {frame_count} frames from {VIDEO_PATH} → {FRAME_DIR}/")
print(f"Video FPS: {fps:.1f}")

### Step 2: Preview Frame 0 and Choose a Prompt Point

We'll display the first frame so you can pick a point to track. Set the `PROMPT_POINT` coordinates below to click on your target object — the `*` marker shows where your prompt lands.

> **Tip:** Adjust `PROMPT_POINT` to target different objects. Use `PROMPT_LABEL = 0` for a *background* point (tells SAM what to exclude).

In [ ]:
# ── PROMPT CONFIG ──
# (x, y) coordinates on frame 0 — adjust these to target your object
PROMPT_POINT = np.array([[330, 220]], dtype=np.float32)  # Centre of the car
PROMPT_LABEL = np.array([1], dtype=np.int32)             # 1 = foreground
# ───────────────────

# Load frame 0 for preview
frame0_path = os.path.join(FRAME_DIR, "00000.jpg")
frame0 = cv2.cvtColor(cv2.imread(frame0_path), cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(frame0)
show_points(PROMPT_POINT, PROMPT_LABEL, ax, marker_size=400)
ax.set_title("Frame 0 — Your Prompt Point (adjust PROMPT_POINT above)", fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()

print(f"Prompt point: {PROMPT_POINT[0]} ({'foreground' if PROMPT_LABEL[0] == 1 else 'background'})")
print(f"Frame size: {frame0.shape[1]}x{frame0.shape[0]}")

### Step 3: Initialise the Video Predictor and Add the Prompt

We build the SAM 2 Video Predictor, point it at our frames directory, and register our prompt on frame 0.

In [ ]:
from sam2.build_sam import build_sam2_video_predictor

# 1. Build the video predictor
video_predictor = build_sam2_video_predictor(SAM2_CONFIG, SAM2_CHECKPOINT, device=device)
print("SAM 2 Video Predictor loaded!")

# 2. Initialise state — this scans the frame directory and builds an internal frame index
inference_state = video_predictor.init_state(video_path=FRAME_DIR)
print(f"Initialised state for {frame_count} frames")

# 3. Add our prompt on frame 0
#    obj_id=1 assigns an identity — you can add multiple objects with different IDs
_, out_obj_ids, out_mask_logits = video_predictor.add_new_points_or_box(
    inference_state=inference_state,
    frame_idx=0,
    obj_id=1,
    points=PROMPT_POINT,
    labels=PROMPT_LABEL,
)

# Preview the initial mask on frame 0
mask_frame0 = (out_mask_logits[0] > 0.0).cpu().numpy().squeeze()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
ax1.imshow(frame0)
show_points(PROMPT_POINT, PROMPT_LABEL, ax1)
ax1.set_title("Prompt Point")
ax1.axis('off')

ax2.imshow(frame0)
show_mask(mask_frame0, ax2, color=[1.0, 0.0, 0.78])
show_points(PROMPT_POINT, PROMPT_LABEL, ax2)
ax2.set_title("SAM 2 Initial Mask (Frame 0)")
ax2.axis('off')

plt.suptitle("Video Predictor — Prompt & Initial Segmentation", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Step 4: Propagate Through the Entire Video

This is the magic step. SAM 2 takes its memory of the object from frame 0 and **propagates forward** through every subsequent frame, producing a mask at each step. No re-prompting needed.

In [ ]:
# Propagate masks through all frames
video_segments = {}

with torch.inference_mode():
    for out_frame_idx, out_obj_ids, out_mask_logits in video_predictor.propagate_in_video(inference_state):
        video_segments[out_frame_idx] = {
            obj_id: (out_mask_logits[i] > 0.0).cpu().numpy().squeeze()
            for i, obj_id in enumerate(out_obj_ids)
        }

print(f"Propagation complete! Got masks for {len(video_segments)} frames.")

In [ ]:
# Preview masks on a few evenly-spaced sample frames
sample_indices = np.linspace(0, len(video_segments) - 1, 6, dtype=int)
frame_files = sorted(os.listdir(FRAME_DIR))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, idx in zip(axes.flat, sample_indices):
    frame_path = os.path.join(FRAME_DIR, frame_files[idx])
    frame_rgb = cv2.cvtColor(cv2.imread(frame_path), cv2.COLOR_BGR2RGB)
    ax.imshow(frame_rgb)
    if idx in video_segments:
        for obj_id, mask in video_segments[idx].items():
            show_mask(mask, ax, color=[1.0, 0.0, 0.78])
    ax.set_title(f"Frame {idx}", fontsize=11)
    ax.axis('off')

plt.suptitle("SAM 2 Video Predictor — Tracked Object Across Frames", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Step 5: Write the Output Video

Overlay the tracking masks onto the original frames and save as an MP4 file.

In [ ]:
OUTPUT_VIDEO = "results/output_sam2_video_predictor.mp4"

frame_files = sorted(os.listdir(FRAME_DIR))
sample_frame = cv2.imread(os.path.join(FRAME_DIR, frame_files[0]))
h, w = sample_frame.shape[:2]

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (w, h))

MASK_COLOR = np.array([255, 0, 200], dtype=np.float32)  # bright pink overlay

for i, fname in enumerate(frame_files):
    frame = cv2.imread(os.path.join(FRAME_DIR, fname))
    overlay = frame.copy()

    if i in video_segments:
        for obj_id, mask in video_segments[i].items():
            overlay[mask] = (
                overlay[mask].astype(np.float32) * 0.5 + MASK_COLOR * 0.5
            ).astype(np.uint8)
            # Draw contour outline for clarity
            contours, _ = cv2.findContours(
                mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
            )
            cv2.drawContours(overlay, contours, -1, (255, 0, 200), 2)

    out.write(overlay)

out.release()
print(f"Output saved to {OUTPUT_VIDEO} ({len(frame_files)} frames @ {fps:.0f} FPS)")

In [ ]:
# Preview a few frames from the output
if os.path.exists(OUTPUT_VIDEO):
    preview_cap = cv2.VideoCapture(OUTPUT_VIDEO)
    total_out = int(preview_cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(1, total_out // 4)

    sample_frames = []
    for i in range(0, total_out, step):
        preview_cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = preview_cap.read()
        if ret:
            sample_frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    preview_cap.release()

    if sample_frames:
        fig, axes = plt.subplots(1, len(sample_frames), figsize=(5 * len(sample_frames), 5))
        if len(sample_frames) == 1:
            axes = [axes]
        for ax, f in zip(axes, sample_frames):
            ax.imshow(f)
            ax.axis('off')
        plt.suptitle("SAM 2 Video Predictor — Output Frames", fontsize=14)
        plt.tight_layout()
        plt.show()

---

## Task 2: Grounding DINO + SAM 2 — Text-Prompted Segmentation

In Notebook 03's Mega-Pipeline, we used **YOLO** to detect objects before passing bounding boxes to SAM 2. YOLO is fast, but it only knows its **fixed 80 COCO classes** — if your object isn't in that list, you're stuck.

**Grounding DINO** replaces YOLO with a **text-conditioned detector**. You type a description like `"coffee mug"` or `"red fire hydrant"`, and Grounding DINO finds it — even if it's never seen that exact category during training.

### How It Works
1. **Text encoder** processes your prompt (`"bird on a branch"`).
2. **Image encoder** processes the image (a Swin Transformer backbone).
3. **Cross-modal decoder** fuses text and image features to predict bounding boxes that match the description.
4. **SAM 2** takes those boxes and produces pixel-perfect masks — just like the Mega-Pipeline.

```
Text: "bird on a branch"  ──→  Grounding DINO  ──→  Bounding Box
                                                          ↓
Image: kookaburra.jpeg     ──→                      SAM 2 Mask
```

This is a fully **open-vocabulary** pipeline: no class lists, no training — just describe what you want.

In [ ]:
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

# 1. Load Grounding DINO (downloads ~700 MB on first run, then cached)
gdino_model_id = "IDEA-Research/grounding-dino-tiny"

gdino_processor = AutoProcessor.from_pretrained(gdino_model_id)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(gdino_model_id).to(device)
print(f"Grounding DINO loaded on {device}!")

# 2. Load SAM 2 Image Predictor (reuse existing checkpoint)
sam2_model = build_sam2(SAM2_CONFIG, SAM2_CHECKPOINT, device=device)
sam2_predictor = SAM2ImagePredictor(sam2_model)
print("SAM 2 Image Predictor loaded!")

In [ ]:
# 3. Load our sample image
image_bgr = cv2.imread("samples/images/kookaburra.jpeg")
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
pil_image = Image.fromarray(image_rgb)

plt.figure(figsize=(10, 7))
plt.imshow(image_rgb)
plt.title("Input Image")
plt.axis('off')
plt.show()

### Run Grounding DINO with a Text Prompt

Type any description of what you want to detect. Grounding DINO will find matching objects and return bounding boxes with confidence scores.

> **Tip:** Try different prompts — `"bird"`, `"eye"`, `"branch"`, `"bird on a branch"`, or even `"animal"`. Separate multiple objects with periods: `"bird . branch . sky"`.

In [ ]:
# ── TEXT PROMPT ── (change this to detect anything!)
TEXT_PROMPT = "bird . branch"
BOX_THRESHOLD = 0.3
# ─────────────────

# 4. Run Grounding DINO
inputs = gdino_processor(images=pil_image, text=TEXT_PROMPT, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = gdino_model(**inputs)

# 5. Post-process: convert model outputs to boxes, scores, and labels
results = gdino_processor.post_process_grounded_object_detection(
    outputs,
    input_ids=inputs.input_ids,
    target_sizes=[pil_image.size[::-1]],  # (height, width)
)[0]

# Filter by confidence threshold (post-process returns all detections)
keep = results["scores"] >= BOX_THRESHOLD
boxes = results["boxes"][keep].cpu().numpy()       # (N, 4) in xyxy format
scores = results["scores"][keep].cpu().numpy()     # (N,)
labels = [results["labels"][i] for i in range(len(results["labels"])) if keep[i]]

print(f'Grounding DINO found {len(boxes)} object(s) for prompt: "{TEXT_PROMPT}"\n')
for i, (box, score, label) in enumerate(zip(boxes, scores, labels)):
    print(f'  [{i+1}] "{label}" — confidence: {score:.2f}  box: [{box[0]:.0f}, {box[1]:.0f}, {box[2]:.0f}, {box[3]:.0f}]')

In [ ]:
# 6. Visualise Grounding DINO detections (boxes only — before SAM)
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(image_rgb)

for box, score, label in zip(boxes, scores, labels):
    show_box(box, ax, label=f"{label} {score:.0%}", color='lime')

ax.set_title(f'Grounding DINO Detections — Prompt: "{TEXT_PROMPT}"', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()

### Refine Detections with SAM 2

Now we feed each Grounding DINO bounding box into SAM 2 to get **pixel-perfect masks** — exactly like the Mega-Pipeline, but with zero-shot text prompts instead of YOLO's fixed class list.

In [ ]:
# 7. Feed each detection box into SAM 2 for pixel-perfect masks
sam_results = []

with torch.inference_mode():
    sam2_predictor.set_image(image_rgb)

    for box, score, label in zip(boxes, scores, labels):
        sam_masks, sam_scores, _ = sam2_predictor.predict(
            box=box,
            multimask_output=False,
        )
        mask = sam_masks[0] > 0.5

        sam_results.append({
            'box': box,
            'label': label,
            'gdino_score': float(score),
            'mask': mask,
            'sam_score': float(sam_scores[0]),
        })

print(f"SAM 2 refined {len(sam_results)} detection(s):\n")
for i, r in enumerate(sam_results):
    print(f'  [{i+1}] "{r["label"]}" — DINO: {r["gdino_score"]:.2f}, SAM quality: {r["sam_score"]:.3f}')

In [ ]:
# 8. Side-by-side comparison: Grounding DINO boxes vs SAM 2 masks
fig, (ax_dino, ax_sam) = plt.subplots(1, 2, figsize=(18, 8))

# Left — Grounding DINO bounding boxes
ax_dino.imshow(image_rgb)
for r in sam_results:
    show_box(r['box'], ax_dino, label=f"{r['label']} {r['gdino_score']:.0%}")
ax_dino.set_title("Grounding DINO (Text-Prompted Boxes)", fontsize=13)
ax_dino.axis('off')

# Right — SAM 2 masks
ax_sam.imshow(image_rgb)
for r in sam_results:
    show_mask(r['mask'], ax_sam)
    show_box(r['box'], ax_sam, label=f"{r['label']} {r['gdino_score']:.0%}", color='white')
ax_sam.set_title("Grounding DINO + SAM 2 (Pixel-Perfect Masks)", fontsize=13)
ax_sam.axis('off')

plt.suptitle(f'Open-Vocabulary Pipeline — Prompt: "{TEXT_PROMPT}"', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### Experiment: Try Your Own Prompts

Go back and change `TEXT_PROMPT` to detect anything in the image — or load a different image entirely. Some ideas to try:

| Prompt | What Happens |
|---|---|
| `"bird"` | Detects the kookaburra |
| `"eye"` | Finds the bird's eye (fine-grained!) |
| `"tree bark"` | Detects the textured surface |
| `"bird . branch . sky"` | Multiple objects in one pass |

This is the power of open-vocabulary detection — **no retraining, no class lists, just language**.

---

## Task 3: Fine-Tuning CLIP for Domain-Specific Classification

In Notebook 03, we used CLIP for zero-shot classification — and it worked remarkably well out of the box. But what if your domain has **specialised categories** that CLIP's generic training didn't cover well?

For example, CLIP might struggle to distinguish between:
- Species of Australian birds
- Types of industrial defects
- Subtle medical imaging findings

**Fine-tuning** adapts CLIP's representations to your domain using a small labelled dataset. We'll fine-tune CLIP's projection heads (keeping the heavy vision and text backbones frozen) — this is fast, data-efficient, and avoids catastrophic forgetting.

### Strategy: Contrastive Fine-Tuning
1. **Freeze** the image encoder and text encoder (they already have excellent features).
2. **Unfreeze** the projection layers that map features into the shared embedding space.
3. **Train** with contrastive loss on domain-specific image–text pairs.
4. **Evaluate** zero-shot performance before and after fine-tuning.

In [ ]:
from transformers import CLIPProcessor, CLIPModel
from datasets import load_dataset

# 1. Load CLIP
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
print("CLIP model loaded!")

# 2. Load a small domain-specific dataset
#    We use the "beans" dataset — 3 classes of bean leaf diseases
#    This simulates a niche domain that CLIP wasn't specifically trained for
dataset = load_dataset("AI-Lab-Makerere/beans", split="train")
val_dataset = load_dataset("AI-Lab-Makerere/beans", split="validation")

# Map class indices to descriptive text labels
class_names = dataset.features["labels"].names  # e.g. ['angular_leaf_spot', 'bean_rust', 'healthy']
text_labels = [f"a photo of a bean leaf with {name.replace('_', ' ')}" for name in class_names]

print(f"\nDataset: {len(dataset)} training images, {len(val_dataset)} validation images")
print(f"Classes: {class_names}")
print(f"Text labels: {text_labels}")

In [ ]:
# 3. Preview some samples from the dataset
fig, axes = plt.subplots(1, 6, figsize=(20, 4))
for i, ax in enumerate(axes):
    sample = dataset[i * (len(dataset) // 6)]
    ax.imshow(sample["image"])
    ax.set_title(class_names[sample["labels"]], fontsize=10)
    ax.axis('off')

plt.suptitle("Bean Leaf Disease Dataset — Sample Images", fontsize=14)
plt.tight_layout()
plt.show()

### Evaluate CLIP's Zero-Shot Baseline (Before Fine-Tuning)

First, let's measure how well CLIP classifies this domain **without any adaptation**. This gives us a baseline to beat.

In [ ]:
# 4. Zero-shot evaluation — BEFORE fine-tuning
def evaluate_clip(model, processor, dataset, text_labels, device, max_samples=200):
    """Evaluate zero-shot classification accuracy."""
    model.eval()
    correct = 0
    total = 0

    # Pre-encode text labels once
    text_inputs = processor(text=text_labels, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        text_out = model.get_text_features(**text_inputs)
        text_features = text_out if isinstance(text_out, torch.Tensor) else text_out.pooler_output
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    # Classify each image
    indices = np.linspace(0, len(dataset) - 1, min(max_samples, len(dataset)), dtype=int)
    for idx in indices:
        sample = dataset[int(idx)]
        image = sample["image"]
        true_label = sample["labels"]

        image_inputs = processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            img_out = model.get_image_features(**image_inputs)
            image_features = img_out if isinstance(img_out, torch.Tensor) else img_out.pooler_output
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        similarity = (image_features @ text_features.T).squeeze(0)
        predicted = similarity.argmax().item()

        if predicted == true_label:
            correct += 1
        total += 1

    return correct / total


baseline_accuracy = evaluate_clip(clip_model, clip_processor, val_dataset, text_labels, device)
print(f"CLIP Zero-Shot Accuracy (BEFORE fine-tuning): {baseline_accuracy:.1%}")

### Fine-Tune CLIP's Projection Layers

We freeze the heavy encoder backbones and only train the lightweight projection heads. This is fast (a few minutes on CPU) and only needs a small dataset.

In [ ]:
# 5. Freeze everything except the projection layers
for param in clip_model.parameters():
    param.requires_grad = False

# Unfreeze visual and text projection heads
for param in clip_model.visual_projection.parameters():
    param.requires_grad = True
for param in clip_model.text_projection.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in clip_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in clip_model.parameters())
print(f"Trainable parameters: {trainable:,} / {total_params:,} ({trainable/total_params:.2%})")

In [ ]:
# 6. Training loop — contrastive fine-tuning
EPOCHS = 3
BATCH_SIZE = 16
LR = 1e-4

optimizer = torch.optim.AdamW(
    [p for p in clip_model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=0.01,
)

clip_model.train()

# Shuffle indices for mini-batch training
indices = np.arange(len(dataset))

for epoch in range(EPOCHS):
    np.random.shuffle(indices)
    epoch_loss = 0.0
    num_batches = 0

    for start in range(0, len(indices), BATCH_SIZE):
        batch_indices = indices[start:start + BATCH_SIZE]

        # Gather batch images and their corresponding text labels
        batch_images = [dataset[int(i)]["image"] for i in batch_indices]
        batch_texts = [text_labels[dataset[int(i)]["labels"]] for i in batch_indices]

        # Process through CLIP
        inputs = clip_processor(
            text=batch_texts,
            images=batch_images,
            return_tensors="pt",
            padding=True,
        ).to(device)

        outputs = clip_model(**inputs, return_loss=True)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    avg_loss = epoch_loss / num_batches
    print(f"  Epoch {epoch + 1}/{EPOCHS} — Loss: {avg_loss:.4f}")

print("\nFine-tuning complete!")

### Evaluate After Fine-Tuning

Let's measure the accuracy again with the same evaluation to see how much our domain adaptation helped.

In [ ]:
# 7. Zero-shot evaluation — AFTER fine-tuning
finetuned_accuracy = evaluate_clip(clip_model, clip_processor, val_dataset, text_labels, device)
print(f"CLIP Zero-Shot Accuracy (BEFORE fine-tuning): {baseline_accuracy:.1%}")
print(f"CLIP Zero-Shot Accuracy (AFTER  fine-tuning): {finetuned_accuracy:.1%}")
print(f"Improvement: {finetuned_accuracy - baseline_accuracy:+.1%}")

In [ ]:
# 8. Visualise predictions on sample validation images
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

sample_indices = np.linspace(0, len(val_dataset) - 1, 4, dtype=int)

# Pre-encode text labels
text_inputs = clip_processor(text=text_labels, return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    text_out = clip_model.get_text_features(**text_inputs)
    text_features = text_out if isinstance(text_out, torch.Tensor) else text_out.pooler_output
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

for col, idx in enumerate(sample_indices):
    sample = val_dataset[int(idx)]
    image = sample["image"]
    true_label = class_names[sample["labels"]]

    # Get predictions
    image_inputs = clip_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        img_out = clip_model.get_image_features(**image_inputs)
        image_features = img_out if isinstance(img_out, torch.Tensor) else img_out.pooler_output
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

    similarity = (image_features @ text_features.T).squeeze(0)
    probs = similarity.softmax(dim=0).cpu().numpy()
    predicted_idx = probs.argmax()
    predicted_label = class_names[predicted_idx]

    # Top row — images with prediction labels
    axes[0, col].imshow(image)
    correct = predicted_label == true_label
    color = 'green' if correct else 'red'
    axes[0, col].set_title(f"True: {true_label}\nPred: {predicted_label}",
                           fontsize=10, color=color)
    axes[0, col].axis('off')

    # Bottom row — probability bar charts
    colors = ['#2ecc71' if i == predicted_idx else '#3498db' for i in range(len(class_names))]
    short_names = [n.replace('_', '\n') for n in class_names]
    axes[1, col].barh(short_names, probs, color=colors)
    axes[1, col].set_xlim(0, 1)
    for i, p in enumerate(probs):
        axes[1, col].text(p + 0.02, i, f"{p:.0%}", va='center', fontsize=9)

plt.suptitle("Fine-Tuned CLIP — Predictions on Validation Images", fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

### Why This Matters

Fine-tuning just the **projection layers** (~0.5% of CLIP's parameters) adapts the model to your domain while preserving its general capabilities. In production, this same technique can be applied to:

- **Medical imaging**: Distinguish between similar-looking pathologies
- **Manufacturing QA**: Detect subtle defect types
- **Ecology**: Classify species within a narrow taxonomic group
- **Retail**: Identify products from your specific catalogue

---

## Recap

| What We Did | Models | Key Takeaway |
|---|---|---|
| **Video Object Tracking** | SAM 2 Video Predictor | Prompt once, track forever — temporal attention propagates masks across frames |
| **Text-Prompted Segmentation** | Grounding DINO + SAM 2 | Describe what you want in plain English — no class lists, no training |
| **Domain-Adapted CLIP** | Fine-tuned CLIP | A tiny amount of fine-tuning goes a long way for specialised domains |

### Key Concepts
- **SAM 2 Video Predictor** maintains a memory bank, enabling consistent object tracking without per-frame prompts.
- **Grounding DINO** is a text-conditioned detector that enables fully **open-vocabulary** pipelines — the successor to fixed-class detectors like YOLO.
- **Projection-layer fine-tuning** is the most efficient way to adapt foundation models — freeze the backbone, train the heads.

These three techniques represent the current state of the art in computer vision: **foundation models** that generalise broadly, combined with **lightweight adaptation** for domain-specific performance.